# LeetCode 1218 — Longest Arithmetic Subsequence of Given Difference bằng Python

Link đề: https://leetcode.com/problems/longest-arithmetic-subsequence-of-given-difference/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def longestSubsequence(self, arr: List[int], difference: int) -> int:
        ...
```

Ghi chú: trả về **độ dài** dãy con dài nhất (không cần chỉ ra dãy) mà hai phần tử kề nhau trong dãy con chênh nhau đúng `difference`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Phân biệt rõ **dãy con (subsequence)** — giữ thứ tự, được bỏ phần tử, không cần liên tiếp — với **mảng con (subarray)** — phải liên tiếp.
2. Xây được **quy hoạch động (Dynamic Programming — DP)** "dãy con kết thúc tại vị trí `i`": `dp[i] = 1 + dp[j]` với `j` là phần tử đứng trước hợp lệ.
3. Tối ưu từ `O(n²)` xuống `O(n)` bằng nhận xét: phần tử đứng trước **bắt buộc** có giá trị `arr[i] − difference` → tra bằng **hash map** (`dict`) theo **giá trị** thay vì theo chỉ số.
4. Biết thêm phiên bản dùng **mảng có offset** khi miền giá trị nhỏ (`[-10^4, 10^4]`).
5. Tránh các lỗi: nhầm dấu `v + d` / `v − d`, sắp xếp mảng trước, dùng `set` khi `difference = 0`, coi như mảng con liên tiếp.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` và số nguyên `difference`. Trả về độ dài của **dãy con dài nhất** của `arr` là **cấp số cộng** với công sai đúng bằng `difference` (tức là mỗi phần tử trong dãy con bằng phần tử ngay trước nó cộng `difference`).

**Dãy con** là dãy thu được từ `arr` bằng cách xoá đi một số (hoặc không) phần tử mà **không đổi thứ tự** các phần tử còn lại.

**Ví dụ 1:** `arr = [1,2,3,4], difference = 1` → `4` (dãy `[1,2,3,4]`).

**Ví dụ 2:** `arr = [1,3,5,7], difference = 1` → `1` (chỉ một phần tử bất kỳ).

**Ví dụ 3:** `arr = [1,5,7,8,5,3,4,2,1], difference = -2` → `4` (dãy `[7,5,3,1]`).

**Ràng buộc:**
- `1 <= arr.length <= 10^5`
- `-10^4 <= arr[i], difference <= 10^4`

In [1]:
import random
import time
from itertools import combinations
from typing import List

examples = [
    ([1, 2, 3, 4], 1, 4),
    ([1, 3, 5, 7], 1, 1),
    ([1, 5, 7, 8, 5, 3, 4, 2, 1], -2, 4),
]
for arr, d, want in examples:
    print(f"arr={arr}, difference={d} -> {want}")

arr=[1, 2, 3, 4], difference=1 -> 4
arr=[1, 3, 5, 7], difference=1 -> 1
arr=[1, 5, 7, 8, 5, 3, 4, 2, 1], difference=-2 -> 4


## 3. Kiến thức nền

### 3.1 Dãy con vs mảng con

- **Mảng con** (subarray): một đoạn liên tiếp `arr[l..r]`.
- **Dãy con** (subsequence): chọn các chỉ số tăng dần `i1 < i2 < ... < ik` bất kỳ.

Trong ví dụ 3, `[7, 5, 3, 1]` lấy từ chỉ số `2, 4, 5, 8` — **không liên tiếp** — nhưng vẫn hợp lệ.

In [2]:
arr = [1, 5, 7, 8, 5, 3, 4, 2, 1]
idx = [2, 4, 5, 8]
print("dãy con:", [arr[i] for i in idx], "| chỉ số tăng dần:", idx == sorted(idx))
print("mảng con arr[2:6]:", arr[2:6])

dãy con: [7, 5, 3, 1] | chỉ số tăng dần: True
mảng con arr[2:6]: [7, 8, 5, 3]


### 3.2 Kiểm tra một dãy có phải cấp số cộng công sai `d`

Một dãy `b` (độ dài ≥ 1) là cấp số cộng công sai `d` nếu `b[k+1] − b[k] == d` với mọi `k`. Dãy 1 phần tử luôn thoả.

In [3]:
def is_ap(b, d):
    return all(b[k + 1] - b[k] == d for k in range(len(b) - 1))


print(is_ap([7, 5, 3, 1], -2), is_ap([1, 2, 4], 1), is_ap([42], 100))

True False True


### 3.3 `dict` và `dict.get(key, default)`

`dict` (hash map) ánh xạ khoá → giá trị, tra cứu/gán trung bình `O(1)`. `d.get(k, 0)` trả về `d[k]` nếu có khoá, ngược lại trả `0` — rất tiện cho DP "chưa thấy thì coi như 0".

In [4]:
best = {}
best[7] = 1
print(best.get(7, 0), best.get(9, 0))
best[5] = best.get(7, 0) + 1
print(best)

1 0
{7: 1, 5: 2}


### 3.4 Mảng với offset (dịch chỉ số)

Khi giá trị nằm trong `[-10^4, 10^4]`, ta có thể dùng list cỡ `20001` và chỉ số `v + 10000` thay cho `dict`. Truy cập list nhanh hơn `dict` một chút và không có băm.

In [5]:
OFF = 10 ** 4
table = [0] * (2 * OFF + 1)
for v in [-10000, 0, 10000]:
    table[v + OFF] += 1
print(table[0], table[OFF], table[2 * OFF], len(table))

1 1 1 20001


## 4. Ý tưởng nền tảng / trực giác

Xét ví dụ 3 với `difference = −2`. Hỏi: "dãy con tốt nhất **kết thúc tại** mỗi phần tử dài bao nhiêu?"

| i | arr[i] | Phần tử trước phải là `arr[i] − d = arr[i] + 2` | Dài nhất kết thúc tại i |
|---|---|---|---|
| 0 | 1 | 3 (chưa xuất hiện) | 1 |
| 1 | 5 | 7 (chưa) | 1 |
| 2 | 7 | 9 (chưa) | 1 |
| 3 | 8 | 10 (chưa) | 1 |
| 4 | 5 | 7 (có, dài 1) | 2 |
| 5 | 3 | 5 (có, dài **2** — lấy cái số 5 ở i=4) | 3 |
| 6 | 4 | 6 (chưa) | 1 |
| 7 | 2 | 4 (có, dài 1) | 2 |
| 8 | 1 | 3 (có, dài 3) | **4** |

Đáp án = max cột cuối = `4`. Điểm mấu chốt: phần tử đứng trước **không phải tuỳ chọn** — nó **bắt buộc** có giá trị `arr[i] − d`. Ta chỉ cần biết "dãy tốt nhất kết thúc bằng **giá trị** đó, ở bên trái" dài bao nhiêu.

## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi dãy con (mọi tập chỉ số), giữ dãy dài nhất là cấp số cộng công sai `d`. Thử độ dài từ lớn đến nhỏ, gặp là dừng.

Số dãy con là `2^n` → chỉ dùng được cho `n ≤ ~15`, nhưng là **trọng tài** tuyệt đối đáng tin.

In [6]:
def longest_brute(arr, d):
    n = len(arr)
    for k in range(n, 0, -1):
        for idx in combinations(range(n), k):
            if is_ap([arr[i] for i in idx], d):
                return k
    return 0   # không xảy ra vì n >= 1


for arr, d, want in examples:
    got = longest_brute(arr, d)
    print(got)
    assert got == want

4
1
4


**Độ phức tạp:** Time `O(2^n · n)`, Space `O(n)`. Với `n = 10^5` thì không tưởng.

**Nút thắt:** ta xây lại cả dãy con từ đầu cho mỗi tổ hợp, trong khi dãy con tốt nhất kết thúc tại `i` chỉ phụ thuộc vào dãy tốt nhất kết thúc tại **một** phần tử trước đó → cấu trúc con tối ưu → DP.

## 6. Bước đột phá

**Định nghĩa trạng thái:** `dp[i]` = độ dài dãy con dài nhất là cấp số cộng công sai `d` và **kết thúc tại chỉ số `i`**.

**Chuyển trạng thái:** dãy kết thúc tại `i` hoặc chỉ có `arr[i]` (dài 1), hoặc nối vào sau một dãy kết thúc tại `j < i` với `arr[j] = arr[i] − d`:

`dp[i] = 1 + max(dp[j] : j < i, arr[j] == arr[i] − d)` (max rỗng = 0).

**Insight tối ưu:** điều kiện chỉ phụ thuộc **giá trị** `arr[j]`. Vậy thay vì lưu theo chỉ số, lưu theo giá trị:

> `best[v]` = độ dài dãy dài nhất (trong phần đã duyệt) **kết thúc bằng giá trị `v`**.

Khi gặp `v = arr[i]`: **nếu** `v − d` đã có trong `best` **thì** `best[v] = best[v − d] + 1`, **ngược lại** `best[v] = 1`.

**Vì sao được ghi đè `best[v]`?** Gặp lại giá trị `v` ở vị trí sau, `best[v − d]` lúc này **không nhỏ hơn** lúc trước (các giá trị trong `best` chỉ tăng theo thời gian), nên giá trị mới luôn ≥ giá trị cũ. Ghi đè không mất thông tin.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP theo chỉ số, `O(n²)`

Cài đặt trực tiếp công thức: với mỗi `i`, quét mọi `j < i`.

In [7]:
def longest_dp_n2(arr, d):
    n = len(arr)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if arr[i] - arr[j] == d:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)


for arr, d, want in examples:
    assert longest_dp_n2(arr, d) == want
print("Cách 2 đúng trên ví dụ; dp của ví dụ 3:")
arr, d, _ = examples[2]
n = len(arr)
dp = [1] * n
for i in range(n):
    for j in range(i):
        if arr[i] - arr[j] == d:
            dp[i] = max(dp[i], dp[j] + 1)
print(dp)

Cách 2 đúng trên ví dụ; dp của ví dụ 3:
[1, 1, 1, 1, 2, 3, 1, 2, 4]


**Độ phức tạp:** Time `O(n²)` — với `n = 10^5` là `5·10^9` phép so sánh, quá chậm. Space `O(n)`.

### 7.2 Cách 3 — Hash map theo giá trị, `O(n)` (lời giải chuẩn)

In [8]:
def longest_hash(arr, d):
    best = {}
    for v in arr:
        best[v] = best.get(v - d, 0) + 1
    return max(best.values())


for arr, d, want in examples:
    assert longest_hash(arr, d) == want
print("Cách 3 đúng trên ví dụ")

Cách 3 đúng trên ví dụ


Chỉ 3 dòng! **Độ phức tạp:** Time `O(n)` trung bình, Space `O(số giá trị khác nhau) ≤ O(n)`.

Lưu ý thứ tự: ta đọc `best[v − d]` **trước** rồi mới ghi `best[v]`. Khi `d = 0`, `v − d = v` nên `best[v] = best[v] + 1` — đếm số lần xuất hiện của `v`, đúng (dãy hằng `v, v, v, ...` là cấp số cộng công sai 0).

### 7.3 Cách 4 — Mảng có offset thay cho `dict`

Giá trị `arr[i]` trong `[-10^4, 10^4]`, nhưng `v − d` có thể ra ngoài khoảng đó (tới `±2·10^4`) — khi ấy chắc chắn chưa xuất hiện, coi là 0. Ta dùng list cỡ `20001`, kiểm tra biên trước khi đọc.

In [9]:
def longest_array(arr, d):
    OFF = 10 ** 4
    best = [0] * (2 * OFF + 1)
    ans = 0
    for v in arr:
        u = v - d
        prev = best[u + OFF] if -OFF <= u <= OFF else 0
        cur = prev + 1
        best[v + OFF] = cur
        if cur > ans:
            ans = cur
    return ans


for arr, d, want in examples:
    assert longest_array(arr, d) == want
print("Cách 4 đúng trên ví dụ")

Cách 4 đúng trên ví dụ


**Độ phức tạp:** Time `O(n)` (cộng `O(V)` khởi tạo, `V = 20001`), Space `O(V)`. Cách này cũng cập nhật `ans` ngay trong vòng lặp thay vì `max(best.values())` ở cuối.

### 7.4 Cách 5 — Tham lam theo điểm bắt đầu (minh hoạ, `O(n²)`)

Với một điểm bắt đầu cố định `s`, nhặt **sớm nhất** phần tử tiếp theo bằng `cur + d` là tối ưu (nhặt muộn hơn chỉ làm phần còn lại ngắn đi). Thử mọi `s` → `O(n²)`. Không nhanh hơn Cách 2, nhưng giúp hiểu vì sao "chỉ cần giá trị".

In [10]:
def longest_greedy_start(arr, d):
    n, ans = len(arr), 0
    for s in range(n):
        cur, length = arr[s], 1
        for k in range(s + 1, n):
            if arr[k] == cur + d:
                cur += d
                length += 1
        ans = max(ans, length)
    return ans


for arr, d, want in examples:
    assert longest_greedy_start(arr, d) == want
print("Cách 5 đúng trên ví dụ")

Cách 5 đúng trên ví dụ


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 3 (`d = −2`). Bảng trạng thái `best` sau mỗi bước:

| Bước | v | v − d | best[v − d] | best[v] mới | ans |
|---|---|---|---|---|---|
| 1 | 1 | 3 | 0 | 1 | 1 |
| 2 | 5 | 7 | 0 | 1 | 1 |
| 3 | 7 | 9 | 0 | 1 | 1 |
| 4 | 8 | 10 | 0 | 1 | 1 |
| 5 | 5 | 7 | 1 | 2 | 2 |
| 6 | 3 | 5 | 2 | 3 | 3 |
| 7 | 4 | 6 | 0 | 1 | 3 |
| 8 | 2 | 4 | 1 | 2 | 3 |
| 9 | 1 | 3 | 3 | **4** | **4** |

Chú ý bước 9: `best[1]` bị ghi đè từ 1 thành 4 — đúng như nhận xét "giá trị mới không nhỏ hơn cũ".

In [11]:
def trace_hash(arr, d):
    best, ans = {}, 0
    print(f"{'bước':>4} {'v':>3} {'v-d':>4} {'best[v-d]':>9} {'best[v]':>7} {'ans':>3}")
    for step, v in enumerate(arr, 1):
        prev = best.get(v - d, 0)
        best[v] = prev + 1
        ans = max(ans, best[v])
        print(f"{step:>4} {v:>3} {v - d:>4} {prev:>9} {best[v]:>7} {ans:>3}   best={best}")
    return ans


trace_hash([1, 5, 7, 8, 5, 3, 4, 2, 1], -2)

bước   v  v-d best[v-d] best[v] ans
   1   1    3         0       1   1   best={1: 1}
   2   5    7         0       1   1   best={1: 1, 5: 1}
   3   7    9         0       1   1   best={1: 1, 5: 1, 7: 1}
   4   8   10         0       1   1   best={1: 1, 5: 1, 7: 1, 8: 1}
   5   5    7         1       2   2   best={1: 1, 5: 2, 7: 1, 8: 1}
   6   3    5         2       3   3   best={1: 1, 5: 2, 7: 1, 8: 1, 3: 3}
   7   4    6         0       1   3   best={1: 1, 5: 2, 7: 1, 8: 1, 3: 3, 4: 1}
   8   2    4         1       2   3   best={1: 1, 5: 2, 7: 1, 8: 1, 3: 3, 4: 1, 2: 2}
   9   1    3         3       4   4   best={1: 4, 5: 2, 7: 1, 8: 1, 3: 3, 4: 1, 2: 2}


4

Muốn biết **dãy** cụ thể (không chỉ độ dài), có thể lưu thêm "cha" theo giá trị. Vì các phần tử đứng trước là `v − d, v − 2d, ...`, chỉ cần biết phần tử cuối và độ dài là dựng lại được:

In [12]:
def reconstruct(arr, d):
    best = {}
    end_val, ans = arr[0], 0
    for v in arr:
        best[v] = best.get(v - d, 0) + 1
        if best[v] > ans:
            ans, end_val = best[v], v
    return [end_val - (ans - 1 - k) * d for k in range(ans)]


for arr, d, _ in examples:
    seq = reconstruct(arr, d)
    print(arr, d, "->", seq)
    assert is_ap(seq, d)

[1, 2, 3, 4] 1 -> [1, 2, 3, 4]
[1, 3, 5, 7] 1 -> [1]
[1, 5, 7, 8, 5, 3, 4, 2, 1] -2 -> [7, 5, 3, 1]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def longestSubsequence(self, arr: List[int], difference: int) -> int:
        best = {}
        ans = 0
        for v in arr:
            cur = best.get(v - difference, 0) + 1
            best[v] = cur
            if cur > ans:
                ans = cur
        return ans


sol = Solution()
for arr, d, want in examples:
    got = sol.longestSubsequence(arr, d)
    print(got)
    assert got == want

4
1
4


## 10. Vì sao đúng

**Bất biến:** sau khi xử lý xong tiền tố `arr[0..i]`, với mọi giá trị `v`, `best[v]` bằng độ dài dãy con hợp lệ dài nhất **nằm trong tiền tố đó** và **kết thúc bằng giá trị `v`** (bằng 0 / không có khoá nếu `v` chưa xuất hiện).

- **Cơ sở:** tiền tố rỗng, `best` rỗng — đúng.
- **Bước:** thêm `v = arr[i]`. Dãy con dài nhất kết thúc tại vị trí `i` hoặc là `[v]`, hoặc là một dãy kết thúc bằng giá trị `v − d` ở vị trí `< i` nối thêm `v`. Theo bất biến, dãy dài nhất loại sau có độ dài `best[v − d]`. Vậy độ dài tốt nhất kết thúc tại `i` là `best[v − d] + 1`.
  Các dãy kết thúc bằng giá trị `v` ở vị trí cũ `j < i` có độ dài `≤ best_cũ[v − d] + 1 ≤ best_mới[v − d] + 1` (giá trị trong `best` không giảm). Nên `max` của cũ và mới chính là giá trị mới → gán đè đúng.
  Khi `d = 0`: `v − d = v`, phép gán dùng giá trị cũ của `best[v]` rồi cộng 1 — vẫn đúng.
- **Kết luận:** đáp án là `max` của mọi `best[v]` (dãy tốt nhất kết thúc ở đâu đó).

## 11. Time / Space Complexity

Gọi `n = len(arr)`, `U` = số giá trị khác nhau, `V = 20001` = kích thước miền giá trị.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(n)` |
| 2. DP theo chỉ số | `O(n²)` | `O(n)` |
| 3. Hash map | `O(n)` trung bình | `O(U)` |
| 4. Mảng offset | `O(n + V)` | `O(V)` |
| 5. Tham lam mọi điểm đầu | `O(n²)` | `O(1)` |

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [14]:
methods = {
    "brute": longest_brute,
    "dp_n2": longest_dp_n2,
    "hash": longest_hash,
    "array": longest_array,
    "greedy": longest_greedy_start,
    "submit": lambda a, d: Solution().longestSubsequence(a, d),
}
edge = [
    ([5], 3, 1),                     # 1 phần tử
    ([2, 2, 2, 2], 0, 4),            # difference = 0: dãy hằng
    ([4, 3, 2, 1], 1, 1),            # đúng giá trị nhưng sai thứ tự
    ([4, 3, 2, 1], -1, 4),
    ([10000, -10000], -20000, 2),    # v - d vượt miền [-1e4, 1e4]
    ([-10000, 10000], 20000, 2),
    ([3, 1, 3, 5, 1, 3, 5, 7], 2, 4),
]
for name, f in methods.items():
    for arr, d, want in examples + edge:
        assert f(arr, d) == want, (name, arr, d)
print("Mọi cách đúng trên ví dụ + edge case")

Mọi cách đúng trên ví dụ + edge case


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Brute force `2^n` chỉ chạy với `n ≤ 10`; với `n` lớn hơn dùng DP `O(n²)` làm trọng tài.

In [15]:
rng = random.Random(1218)
for _ in range(300):
    n = rng.randint(1, 10)
    arr = [rng.randint(-4, 4) for _ in range(n)]
    d = rng.randint(-3, 3)
    want = longest_brute(arr, d)
    for name, f in methods.items():
        assert f(arr, d) == want, (name, arr, d)
for _ in range(200):
    n = rng.randint(1, 200)
    arr = [rng.randint(-30, 30) for _ in range(n)]
    d = rng.randint(-5, 5)
    want = longest_dp_n2(arr, d)
    for name in ["hash", "array", "greedy", "submit"]:
        assert methods[name](arr, d) == want, name
print("500 test ngẫu nhiên khớp trọng tài")

500 test ngẫu nhiên khớp trọng tài


### 12.3 Test giới hạn `n = 10^5`

In [16]:
big_inc = list(range(-10000, 10000)) * 5          # 1e5 phần tử
big_rand = [rng.randint(-10000, 10000) for _ in range(10 ** 5)]
for name in ["hash", "array", "submit"]:
    f = methods[name]
    t0 = time.perf_counter()
    r1 = f(big_inc, 1)
    r2 = f(big_rand, 7)
    r3 = f([7] * 10 ** 5, 0)
    dt = time.perf_counter() - t0
    assert r1 == 20000
    assert r3 == 10 ** 5
    print(f"{name:7s} r1={r1} r2={r2} r3={r3} trong {dt * 1000:.0f} ms")
assert methods["hash"](big_rand, 7) == methods["array"](big_rand, 7)

hash    r1=20000 r2=14 r3=100000 trong 22 ms
array   r1=20000 r2=14 r3=100000 trong 42 ms
submit  r1=20000 r2=14 r3=100000 trong 39 ms


Giải thích `r1`: mảng `[-10000..9999]` lặp 5 lần, công sai 1. Dãy con có thể đi `-10000 → ... → 9999` trong lần lặp đầu (dài 20000), không thể nối dài hơn vì sau `9999` cần `10000` không có trong mảng. Vậy `r1 = 20000`.

In [17]:
assert Solution().longestSubsequence(big_inc, 1) == 20000
print("r1 = 20000 đúng như phân tích")

r1 = 20000 đúng như phân tích


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhầm dấu: tra `v + d` thay vì `v − d`

In [18]:
def wrong_sign(arr, d):
    best = {}
    for v in arr:
        best[v] = best.get(v + d, 0) + 1
    return max(best.values())


print("sai :", wrong_sign([1, 2, 3, 4], 1))     # 1
print("đúng:", longest_hash([1, 2, 3, 4], 1))   # 4

sai : 1
đúng: 4


**Vì sao sai:** dãy `... , u, v` thoả `v − u = d` ⇒ phần tử đứng **trước** là `u = v − d`. Tra `v + d` là tìm phần tử đứng **sau** — mà nó chưa được duyệt tới. (Hàm sai trên thực chất giải bài với công sai `−d`.) **Sửa:** `best.get(v - d, 0)`.

### Lỗi 2 — Sắp xếp mảng trước cho "dễ"

In [19]:
def wrong_sort(arr, d):
    return longest_hash(sorted(arr), d)


print("sai :", wrong_sort([4, 3, 2, 1], 1))     # 4 — nhưng thứ tự gốc giảm dần!
print("đúng:", longest_hash([4, 3, 2, 1], 1))   # 1

sai : 4
đúng: 1


**Vì sao sai:** dãy con phải **giữ thứ tự** gốc. Sắp xếp tạo ra những dãy không tồn tại trong `arr`. **Sửa:** duyệt đúng thứ tự gốc.

### Lỗi 3 — Dùng `set` thay cho `dict` (chỉ nhớ "đã thấy", không nhớ độ dài)

In [20]:
def wrong_set(arr, d):
    seen, run, ans = set(), 0, 0
    for v in arr:
        run = run + 1 if (v - d) in seen else 1     # tưởng "đang chạy" một dãy duy nhất
        seen.add(v)
        ans = max(ans, run)
    return ans


arr = [1, 5, 7, 8, 5, 3, 4, 2, 1]
print("sai :", wrong_set(arr, -2))     # đếm "run" lẫn lộn giữa các dãy khác nhau
print("đúng:", longest_hash(arr, -2))
print("sai :", wrong_set([1, 5, 2, 6], 1), "| đúng:", longest_hash([1, 5, 2, 6], 1), "(đếm lố: nối 6 vào sau dãy 1,2)")

sai : 3
đúng: 4
sai : 3 | đúng: 2 (đếm lố: nối 6 vào sau dãy 1,2)


**Vì sao sai:** có thể có **nhiều dãy song song** đang "dở dang" (ở ví dụ 3: dãy `7,5,3,...` và dãy `4,2,...` đan xen). Một biến `run` duy nhất không phân biệt được chúng. Cần nhớ độ dài **cho từng giá trị kết thúc** → `dict`. **Sửa:** Cách 3.

### Lỗi 4 — Nhầm thành mảng con liên tiếp

In [21]:
def wrong_contiguous(arr, d):
    ans = run = 1
    for i in range(1, len(arr)):
        run = run + 1 if arr[i] - arr[i - 1] == d else 1
        ans = max(ans, run)
    return ans


arr = [1, 5, 7, 8, 5, 3, 4, 2, 1]
print("sai :", wrong_contiguous(arr, -2))   # chỉ thấy đoạn liên tiếp [5, 3]
print("đúng:", longest_hash(arr, -2))

sai : 2
đúng: 4


**Vì sao sai:** đề nói **subsequence**, được bỏ qua phần tử ở giữa. **Sửa:** DP theo giá trị.

### Lỗi 5 — Gọi `max(best.values())` bên trong vòng lặp

Đúng kết quả nhưng `O(n · U)` — có thể chậm tới `10^5 × 2·10^4`. Hãy giữ biến `ans` và cập nhật ngay (như code nộp bài) hoặc gọi `max` một lần ở cuối.

## 14. Mẹo & nhận dạng pattern

- "Dãy con dài nhất thoả quan hệ giữa hai phần tử **kề nhau**" → DP "kết thúc tại `i`".
- Nếu phần tử đứng trước được **xác định duy nhất về giá trị** (ở đây `v − d`) → DP theo **giá trị** bằng `dict`, `O(n)`.
- Nếu phần tử trước chỉ cần thoả bất đẳng thức (như LIS: `arr[j] < arr[i]`) → không tra trực tiếp được, cần cấu trúc khác (binary search / Fenwick).
- Nếu công sai **không cho trước** (LeetCode 1027) → trạng thái phải có thêm công sai: `dp[i][diff]`, `O(n²)`.
- Miền giá trị nhỏ → mảng offset thay `dict`.

## 15. Biến thể

1. **LeetCode 1027 — Longest Arithmetic Subsequence:** không cho `difference`; tìm dãy cấp số cộng dài nhất với công sai bất kỳ → `dp[i]` là `dict` công sai → độ dài.
2. **Dãy con cấp số nhân** công bội `r` cho trước: tra `v / r` (cẩn thận chia không hết, `r = 0`).
3. **Đếm số dãy con** cấp số cộng công sai `d` có độ dài ≥ 2: thay `max` bằng tổng — `cnt[v] += cnt[v − d] + 1`.
4. **Longest Consecutive Sequence (LeetCode 128):** không cần giữ thứ tự (tập hợp) → dùng `set`.

In [22]:
def longest_any_difference(arr):
    """Biến thể 1 (LC 1027): công sai tuỳ ý, O(n^2)."""
    n = len(arr)
    dp = [dict() for _ in range(n)]
    ans = 1
    for i in range(n):
        for j in range(i):
            diff = arr[i] - arr[j]
            dp[i][diff] = dp[j].get(diff, 1) + 1
            ans = max(ans, dp[i][diff])
    return ans


print(longest_any_difference([3, 6, 9, 12]), longest_any_difference([9, 4, 7, 2, 10]), longest_any_difference([20, 1, 15, 3, 10, 5, 8]))

4 3 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi dãy con | `O(2^n·n)` | `O(n)` | trọng tài `n ≤ 12` |
| 2. DP chỉ số | `dp[i] = 1 + max dp[j]` | `O(n²)` | `O(n)` | hiểu công thức, `n ≤ 5000` |
| 3. Hash map giá trị | `best[v] = best[v−d] + 1` | `O(n)` | `O(U)` | **nộp bài** |
| 4. Mảng offset | như 3, dùng list | `O(n+V)` | `O(V)` | miền giá trị nhỏ, cần nhanh |
| 5. Tham lam điểm đầu | nhặt sớm nhất | `O(n²)` | `O(1)` | minh hoạ trực giác |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test.

### Bài 1 (đoán output)

`Solution().longestSubsequence([3, 0, -3, 4, -6, 7, -9], -3)` bằng bao nhiêu?

### Bài 2 (tính tay)

Với `arr = [1, 1, 2, 2, 3]`, `d = 1`: lập bảng `best` sau từng bước như mục 8. Đáp án?

### Bài 3 (điền TODO) — đếm số dãy con cấp số cộng (độ dài ≥ 2)

In [23]:
def count_ap_subsequences(arr, d):
    """Đếm số dãy con (theo tập chỉ số) có độ dài >= 2 là cấp số cộng công sai d."""
    cnt = {}      # cnt[v] = số dãy con (độ dài >= 1) kết thúc bằng giá trị v
    total = 0
    for v in arr:
        # TODO: số dãy độ dài >= 2 kết thúc tại phần tử này = cnt.get(v - d, 0)
        #       cập nhật total và cnt[v]
        raise NotImplementedError
    return total

# print(count_ap_subsequences([1, 2, 3], 1))   # mong đợi 3: [1,2], [2,3], [1,2,3]

### Bài 4 (tự viết lại)

Viết lại lời giải trả về **chính dãy con** (list giá trị) dài nhất, nếu nhiều dãy cùng độ dài thì trả dãy **kết thúc sớm nhất** trong mảng.

In [24]:
def longest_sequence_todo(arr, d):
    # TODO
    raise NotImplementedError

# print(longest_sequence_todo([1, 5, 7, 8, 5, 3, 4, 2, 1], -2))   # mong đợi [7, 5, 3, 1]

### Bài 5 (mini challenge) — cho phép bỏ qua giá trị

Tìm dãy con dài nhất mà hiệu hai phần tử kề nhau thuộc tập `{d1, d2}` cho trước (ví dụ `{1, 2}`). Gợi ý: `best[v] = 1 + max(best[v − d1], best[v − d2])`.

In [25]:
def longest_two_diffs_todo(arr, d1, d2):
    # TODO
    raise NotImplementedError

# print(longest_two_diffs_todo([1, 3, 4, 2, 6], 1, 2))   # mong đợi 4: [1, 3, 4, 6]

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: phần tử trước của `v` là `v + 3`. Mức 2: theo dõi hai "chuỗi": `3, 0, −3, −6, −9` và `4, 7` (công sai sai dấu). Mức 3: chuỗi đầu dài 5.

**Bài 2.** Mức 1: `best[1]` lên 1, rồi lại 1 (vì `best[0]` không có). Mức 2: `best[2] = best[1] + 1 = 2`, gặp 2 lần nữa vẫn 2. Mức 3: `best[3] = 3` → đáp án 3.

**Bài 3.** Mức 1: `k = cnt.get(v − d, 0)`. Mức 2: `total += k`. Mức 3: `cnt[v] = cnt.get(v, 0) + k + 1` (cộng dồn vì nhiều vị trí có cùng giá trị đều là điểm kết thúc khác nhau).

**Bài 4.** Mức 1: lưu `best` như cũ. Mức 2: chỉ cập nhật `(ans, end_val)` khi `best[v] > ans` (lớn hơn **chặt**) để giữ dãy kết thúc sớm nhất. Mức 3: dựng ngược `end_val, end_val − d, ...`.

**Bài 5.** Mức 1: vẫn duyệt trái → phải. Mức 2: dùng `best.get(v − d1, 0)` và `best.get(v − d2, 0)`. Mức 3: nếu `d1 == d2` công thức vẫn đúng.

## 19. Đáp án tham khảo

In [26]:
# Bài 1, 2
print(Solution().longestSubsequence([3, 0, -3, 4, -6, 7, -9], -3))
assert Solution().longestSubsequence([3, 0, -3, 4, -6, 7, -9], -3) == 5
trace_hash([1, 1, 2, 2, 3], 1)
assert Solution().longestSubsequence([1, 1, 2, 2, 3], 1) == 3

5
bước   v  v-d best[v-d] best[v] ans
   1   1    0         0       1   1   best={1: 1}
   2   1    0         0       1   1   best={1: 1}
   3   2    1         1       2   2   best={1: 1, 2: 2}
   4   2    1         1       2   2   best={1: 1, 2: 2}
   5   3    2         2       3   3   best={1: 1, 2: 2, 3: 3}


In [27]:
def count_ap_subsequences(arr, d):
    cnt, total = {}, 0
    for v in arr:
        k = cnt.get(v - d, 0)
        total += k
        cnt[v] = cnt.get(v, 0) + k + 1
    return total


def longest_sequence(arr, d):
    best, ans, end_val = {}, 0, None
    for v in arr:
        best[v] = best.get(v - d, 0) + 1
        if best[v] > ans:
            ans, end_val = best[v], v
    return [end_val - (ans - 1 - k) * d for k in range(ans)]


def longest_two_diffs(arr, d1, d2):
    best, ans = {}, 0
    for v in arr:
        cur = 1 + max(best.get(v - d1, 0), best.get(v - d2, 0))
        best[v] = max(best.get(v, 0), cur)
        ans = max(ans, cur)
    return ans


def brute_count(arr, d):
    n = len(arr)
    return sum(is_ap([arr[i] for i in idx], d) for k in range(2, n + 1) for idx in combinations(range(n), k))


def brute_two(arr, d1, d2):
    n, best = len(arr), 1
    for k in range(1, n + 1):
        for idx in combinations(range(n), k):
            b = [arr[i] for i in idx]
            if all(b[t + 1] - b[t] in (d1, d2) for t in range(k - 1)):
                best = max(best, k)
    return best


assert count_ap_subsequences([1, 2, 3], 1) == 3
assert longest_sequence([1, 5, 7, 8, 5, 3, 4, 2, 1], -2) == [7, 5, 3, 1]
assert longest_two_diffs([1, 3, 4, 2, 6], 1, 2) == 4
rng = random.Random(99)
for _ in range(200):
    arr = [rng.randint(-3, 3) for _ in range(rng.randint(1, 9))]
    d, d1, d2 = rng.randint(-2, 2), rng.randint(-2, 2), rng.randint(-2, 2)
    assert count_ap_subsequences(arr, d) == brute_count(arr, d)
    seq = longest_sequence(arr, d)
    assert len(seq) == longest_brute(arr, d) and is_ap(seq, d)
    assert longest_two_diffs(arr, d1, d2) == brute_two(arr, d1, d2)
print("Đáp án bài 3, 4, 5 đúng")

Đáp án bài 3, 4, 5 đúng


Ghi chú Bài 5: dùng `max(best.get(v, 0), cur)` thay vì gán đè, vì với hai công sai, tính chất "giá trị mới luôn ≥ cũ" vẫn đúng (mọi `best` đều không giảm), nhưng viết `max` cho an toàn khi biến thể phức tạp hơn.

## 20. Template nhớ nhanh + Checklist trước khi submit

```python
best, ans = {}, 0
for v in arr:                          # đúng thứ tự gốc, KHÔNG sort
    cur = best.get(v - d, 0) + 1       # phần tử TRƯỚC là v - d
    best[v] = cur                      # ghi đè an toàn (không giảm)
    ans = max(ans, cur)
return ans
```

**Checklist:**
- [ ] Tra `v − difference` (không phải `v + difference`)?
- [ ] Giữ nguyên thứ tự mảng?
- [ ] `difference = 0` cho đúng số lần xuất hiện lớn nhất?
- [ ] Không gọi `max(best.values())` trong vòng lặp?
- [ ] Mảng 1 phần tử trả `1`?
- [ ] Nếu dùng mảng offset: kiểm tra `v − d` có nằm trong miền trước khi đọc?